# Day 4 — Structured Data Ingestion Lab
Loads plans/claims CSVs, cleans them, loads into SQLite, and runs the 5 required SQL queries.

In [15]:
import sqlite3
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"
DB_PATH = Path.cwd().parent / "coverage.db"

plans = pd.read_csv(DATA_DIR / "plans.csv")
claims = pd.read_csv(DATA_DIR / "claims.csv")

In [16]:
plans.info()
print(plans.head())
claims.info()
print(claims.head())

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   plan_id            3 non-null      str  
 1   plan_name          3 non-null      str  
 2   monthly_premium    3 non-null      int64
 3   annual_deductible  3 non-null      int64
 4   copay_pct          3 non-null      int64
 5   coverage_type      3 non-null      str  
 6   network_tier       3 non-null      str  
dtypes: int64(3), str(4)
memory usage: 300.0 bytes
  plan_id   plan_name  monthly_premium  annual_deductible  copay_pct  \
0    P101    Gold PPO              500               2000         10   
1    P102  Silver HMO              300               1500         20   
2    P103  Bronze HMO              150               1000         30   

  coverage_type network_tier  
0           PPO         Gold  
1           HMO       Silver  
2           HMO       Bronze  
<class 'pandas.DataFrame'>
RangeI

In [17]:
print("plans duplicates:", plans.duplicated().sum())
print("claims duplicates:", claims.duplicated().sum())

plans = plans.drop_duplicates(subset="plan_id", keep="first")
claims = claims.drop_duplicates(subset="claim_id", keep="first")

plans duplicates: 0
claims duplicates: 0


In [18]:
print("plans duplicates:", plans.duplicated().sum())
print("claims duplicates:", claims.duplicated().sum())

plans = plans.drop_duplicates(subset="plan_id", keep="first")
claims = claims.drop_duplicates(subset="claim_id", keep="first")

plans duplicates: 0
claims duplicates: 0


In [19]:
plans["monthly_premium"] = pd.to_numeric(plans["monthly_premium"], errors="coerce")
plans["annual_deductible"] = pd.to_numeric(plans["annual_deductible"], errors="coerce")
plans["copay_pct"] = pd.to_numeric(plans["copay_pct"], errors="coerce")

claims["claim_amount"] = pd.to_numeric(claims["claim_amount"], errors="coerce")
claims["date_filed"] = pd.to_datetime(claims["date_filed"], errors="coerce")

plans = plans.dropna(subset=["plan_id", "monthly_premium", "annual_deductible", "copay_pct"])
claims = claims.dropna(subset=["claim_id", "member_id", "plan_id", "claim_amount"])

print(plans.dtypes)
print(claims.dtypes)

plan_id                str
plan_name              str
monthly_premium      int64
annual_deductible    int64
copay_pct            int64
coverage_type          str
network_tier           str
dtype: object
claim_id                   str
member_id                  str
plan_id                    str
procedure                  str
claim_amount             int64
status                     str
date_filed      datetime64[us]
dtype: object


## Load into SQLite

In [20]:
conn = sqlite3.connect(DB_PATH)
plans.to_sql("plans", conn, if_exists="replace", index=False)
claims.to_sql("claims", conn, if_exists="replace", index=False)
conn.commit()
conn.close()
print(f"Loaded {len(plans)} plans and {len(claims)} claims into {DB_PATH}")

Loaded 3 plans and 5 claims into /Users/utkarshmavi/Desktop/ABTalks - AI Cohort/Daily Task/coverage.db


## SQL Queries — 5 member questions

In [21]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

def run(title, sql):
    print(f"\n{title}")
    cur.execute(sql)
    cols = [d[0] for d in cur.description]
    print(" | ".join(cols))
    for row in cur.fetchall():
        print(" | ".join(str(v) for v in row))

In [22]:
run("Q1: Deductible on Gold PPO", """
    SELECT plan_name, annual_deductible
    FROM plans
    WHERE plan_name = 'Gold PPO';
""")



Q1: Deductible on Gold PPO
plan_name | annual_deductible
Gold PPO | 2000


In [23]:
run("Q2: Pending claim count for M1001", """
    SELECT COUNT(*) AS pending_count
    FROM claims
    WHERE member_id = 'M1001' AND status = 'Pending';
""")


Q2: Pending claim count for M1001
pending_count
1


In [24]:
run("Q3: Plans with monthly premium under $400", """
    SELECT plan_name, monthly_premium
    FROM plans
    WHERE monthly_premium < 400
    ORDER BY monthly_premium;
""")


Q3: Plans with monthly premium under $400
plan_name | monthly_premium
Bronze HMO | 150
Silver HMO | 300


In [25]:
run("Q4: Claims with plan details (JOIN)", """
    SELECT c.claim_id, c.member_id, c.procedure, c.claim_amount, c.status, p.plan_name, p.network_tier
    FROM claims c
    JOIN plans p ON c.plan_id = p.plan_id
    ORDER BY c.claim_id;
""")


Q4: Claims with plan details (JOIN)
claim_id | member_id | procedure | claim_amount | status | plan_name | network_tier
C1001 | M1001 | X-ray | 250 | Pending | Gold PPO | Gold
C1002 | M1001 | Surgery | 1200 | Approved | Gold PPO | Gold
C1003 | M1002 | X-ray | 150 | Denied | Silver HMO | Silver
C1004 | M1002 | Surgery | 900 | Approved | Silver HMO | Silver
C1005 | M1003 | X-ray | 50 | Pending | Bronze HMO | Bronze


In [26]:
run("Q5: Top procedures by claim count", """
    SELECT procedure, COUNT(*) AS claim_count, SUM(claim_amount) AS total_claimed
    FROM claims
    GROUP BY procedure
    ORDER BY claim_count DESC, total_claimed DESC
    LIMIT 5;
""")


Q5: Top procedures by claim count
procedure | claim_count | total_claimed
X-ray | 3 | 450
Surgery | 2 | 2100


In [27]:
conn.close()